# Step 4 — Retrieval

This notebook defines and validates `retrieve_context()`, the retriever function used in all
subsequent steps (fine-tuning, evaluation).

**No files are saved here.** The output of this notebook is the validated function itself.

**Two behaviours:**
- **Test-time** (`chunk_id=None`): tweet is not in the index → return the k closest chunks directly.
- **Train-time** (`chunk_id=int`): tweet IS in the index → fetch k+1, skip the entry whose id
  matches `chunk_id` exactly, return k. Self-exclusion is by id, not by score rank.

**Prerequisite:** `rag_step3_index.ipynb` must have been run, producing:
```
index/roberta-base_IsHate.faiss
index/documents.json
```

## 1. Imports

In [16]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

## 2. Configuration

In [17]:
# Using a single model for all retrieval for simplicity.
# Future improvement: use the IHC-trained model for IHC data and the
# ISHate-trained model for ISHate data separately.
# Even further: run all 6 models and compare retrieval quality.
MODEL_NAME   = "roberta-base_IsHate"
HF_ID        = "roberta-base"
WEIGHTS_PATH = "../weights/roberta-base_ISHate"
INDEX_PATH   = f"index/{MODEL_NAME}.faiss"
K            = 3
BATCH_SIZE   = 64
MAX_LENGTH   = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Model  : {MODEL_NAME}")
print(f"k      : {K}")

Device : cpu
Model  : roberta-base_IsHate
k      : 3


## 3. Load Model, Index & Documents

In [18]:
# Tokenizer loaded from HuggingFace hub — local tokenizer.json was saved with a newer
# tokenizers library version and cannot be parsed by the installed version (same issue as step 3).
tokenizer = AutoTokenizer.from_pretrained(HF_ID)

full_model = AutoModelForSequenceClassification.from_pretrained(WEIGHTS_PATH)
model = full_model.base_model.eval().to(device)
del full_model

index = faiss.read_index(INDEX_PATH)
print(f"Index loaded — {index.ntotal:,} vectors")

with open("index/documents.json") as f:
    documents = json.load(f)
print(f"Documents loaded — {len(documents):,} entries")

Index loaded — 64,796 vectors
Documents loaded — 64,796 entries


## 4. Encoding Function

Identical to step 3. Extracts the `[CLS]` embedding from the last hidden state.

In [19]:
def encode(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch  = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_vecs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls_vecs)
    return np.vstack(all_vecs).astype("float32")

## 5. Retrievers

Three retrieval strategies — all share the same self-exclusion logic (by `chunk_id`, not by score).

| Function | Returns |
|---|---|
| `retrieve_top_k` | exactly k results |
| `retrieve_by_threshold` | all results with score ≥ threshold |
| `retrieve_top_k_above_threshold` | results with score ≥ threshold, capped at k |

In [20]:
def _encode_and_search(tweet_text, fetch):
    vec = encode([tweet_text], model, tokenizer, batch_size=1)
    faiss.normalize_L2(vec)
    scores, ids = index.search(vec, fetch)
    return scores[0], ids[0]


def retrieve_top_k(tweet_text, chunk_id=None, k=K):
    """
    Always returns exactly k neighbors.

    tweet_text : raw tweet string (no [hate]/[not hate] prefix).
    chunk_id   : the tweet's own chunk_id (int) to exclude self at train time.
                 Pass None at test time — the tweet is not in the index.
    Returns    : list of k (text, score) tuples.
    """
    fetch = k + 1 if chunk_id is not None else k
    scores, ids = _encode_and_search(tweet_text, fetch)

    results = []
    for score, cid in zip(scores, ids):
        if cid == -1:
            continue
        if chunk_id is not None and cid == chunk_id:  # exclude by id, NOT by score
            continue
        results.append((documents[str(cid)], float(score)))
        if len(results) == k:
            break
    return results


def retrieve_by_threshold(tweet_text, threshold, chunk_id=None):
    """
    Returns all neighbors whose similarity score is >= threshold, with no upper limit.

    Since k is unbounded, we fetch all vectors in one shot (index.ntotal) when a
    chunk_id is given, so the self-match can always be filtered out regardless of rank.

    tweet_text : raw tweet string (no [hate]/[not hate] prefix).
    threshold  : minimum cosine similarity score (float, e.g. 0.95).
    chunk_id   : the tweet's own chunk_id (int) to exclude self at train time.
                 Pass None at test time.
    Returns    : list of (text, score) tuples with score >= threshold, ordered by score desc.
    """
    fetch = index.ntotal if chunk_id is not None else index.ntotal
    scores, ids = _encode_and_search(tweet_text, fetch)

    results = []
    for score, cid in zip(scores, ids):
        if cid == -1 or score < threshold:
            break  # FAISS returns results sorted by score desc — safe to stop early
        if chunk_id is not None and cid == chunk_id:
            continue
        results.append((documents[str(cid)], float(score)))
    return results


def retrieve_top_k_above_threshold(tweet_text, threshold, chunk_id=None, k=K):
    """
    Returns neighbors with score >= threshold, capped at k results.

    Combines the two strategies: quality filter (threshold) + quantity cap (k).
    Useful when you want at most k results but only if they are genuinely similar.

    tweet_text : raw tweet string (no [hate]/[not hate] prefix).
    threshold  : minimum cosine similarity score (float, e.g. 0.95).
    chunk_id   : the tweet's own chunk_id (int) to exclude self at train time.
                 Pass None at test time.
    k          : maximum number of results to return.
    Returns    : list of (text, score) tuples with score >= threshold, len <= k.
    """
    fetch = k + 1 if chunk_id is not None else k
    scores, ids = _encode_and_search(tweet_text, fetch)

    results = []
    for score, cid in zip(scores, ids):
        if cid == -1 or score < threshold:
            break
        if chunk_id is not None and cid == chunk_id:
            continue
        results.append((documents[str(cid)], float(score)))
        if len(results) == k:
            break
    return results

## 6. Validation — Test-time Behaviour

Queries without a `chunk_id` (simulating the test set, where tweets are not in the index).
Expected:
- Hate tweets → hateful, semantically relevant neighbors.
- "i dont like tomatoes" → weak, unrelated neighbors (low semantic match).
- "i dont like neggers" → hateful neighbors.

In [25]:
THRESHOLD = 0.95

test_queries = [
    ("HATE",     "jews are now in full control : canadian man goes to jail for posting on white nationalist websites"),
    ("HATE",     "i dont like neggers"),
    ("NOT HATE", "i dont like tomatoes"),
    ("NOT HATE", "Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman s fiance , mom ."),
]

sep = "=" * 70

for label, tweet in test_queries:
    print(f"\n{sep}")
    print(f"[{label}] {tweet}")

    print(f"  -- retrieve_top_k (k={K}) --")
    for text, score in retrieve_top_k(tweet, chunk_id=None):
        print(f"    {score:.4f}  {text[:90]}")

    # This print is commented since it produces most of the time hundreds of results. Uncomment it to test this function
    #print(f"  -- retrieve_by_threshold (threshold={THRESHOLD}) --")
    #hits = retrieve_by_threshold(tweet, threshold=THRESHOLD, chunk_id=None)
    #if hits:
    #    for text, score in hits:
    #        print(f"    {score:.4f}  {text[:90]}")
    #else:
     #   print("    (no results above threshold)")

    print(f"  -- retrieve_top_k_above_threshold (threshold={THRESHOLD}, k={10}) --")
    hits = retrieve_top_k_above_threshold(tweet, threshold=THRESHOLD, chunk_id=None, k=10)
    if hits:
        for text, score in hits:
            print(f"    {score:.4f}  {text[:90]}")
    else:
        print("    (no results above threshold)")


[HATE] jews are now in full control : canadian man goes to jail for posting on white nationalist websites
  -- retrieve_top_k (k=3) --


Encoding: 100%|██████████| 1/1 [00:00<00:00, 11.22it/s]


    0.9984  [hate] : they hate you because the jew controlled mainstream media is brainwashing them . 
    0.9984  [hate] white people in britain resisting propaganda to racially miscegenate  figures show 
    0.9982  [hate] japan is able to keep clean his country from refugees . they accept positively from
  -- retrieve_top_k_above_threshold (threshold=0.95, k=10) --


Encoding: 100%|██████████| 1/1 [00:00<00:00, 25.22it/s]


    0.9984  [hate] : they hate you because the jew controlled mainstream media is brainwashing them . 
    0.9984  [hate] white people in britain resisting propaganda to racially miscegenate  figures show 
    0.9982  [hate] japan is able to keep clean his country from refugees . they accept positively from
    0.9982  [hate] there is no hate . what they want to likely root out is any form of white european 
    0.9982  [hate] there is no hate . what they want to doubtlessly root out is any form of white euro
    0.9982  [hate] is not a white supremacist organization . theyre promoting the assertion of white i
    0.9982  [hate]  uk :  " this flat is for blacks only " " : shocking racist campaign to force white
    0.9982  [hate] don't go to #atlanta article and commentary about the dying white race . victims of
    0.9982  [hate] japan is able to keep clean his country from refugees . they accept intensely from 
    0.9982  [hate] the civil rights movement was born out of the sublime 

...

    0.9912  [hate] note that all babies are white .
    0.9907  [hate] pull children are not for rent
    0.9906  [hate] you are anti white
    0.9906  [hate] two men do not start a family
    0.9905  [hate] why do you hate white people ?
    0.9905  [hate] you go guys  white power
    0.9904  [hate] you all liberals care about is diversity
    0.9903  [hate] they eat white babies too
    0.9902  [hate] good job kill americans
    0.9901  [hate] careful don't feed the tribesman

[NOT HATE] Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman s fiance , mom .
  -- retrieve_top_k (k=3) --


Encoding: 100%|██████████| 1/1 [00:00<00:00, 23.74it/s]


    0.9893  [not hate] Ariel Laurie , Israeli Soldier , Grabs Toddler in Australia Kidnaps 6 year old 
    0.9890  [not hate] -- A man wanted for attempted murder in New York City was killed Friday afterno
    0.9887  [not hate] 32-year-old Jason `` Jay '' Tate was found with Kelsey Sue Stahl 's car .
  -- retrieve_top_k_above_threshold (threshold=0.95, k=10) --


Encoding: 100%|██████████| 1/1 [00:00<00:00, 12.70it/s]

    0.9893  [not hate] Ariel Laurie , Israeli Soldier , Grabs Toddler in Australia Kidnaps 6 year old 
    0.9890  [not hate] -- A man wanted for attempted murder in New York City was killed Friday afterno
    0.9887  [not hate] 32-year-old Jason `` Jay '' Tate was found with Kelsey Sue Stahl 's car .
    0.9887  [not hate] - A Memphis teen is behind bars , charged with kidnapping and raping a 15 year 
    0.9885  [not hate] Read more : Charges in deadly wreck that killed 5 motorcyclists - LancasterOnli
    0.9884  [not hate] Jeremy Moore was shot and killed at an Oklahoma City apartment complex while de
    0.9882  [not hate] Tamir Hamilton , sadistic rapist/murder .
    0.9880  [not hate] Jeffrey McAdam was fatally shot at a rest stop along Interstate 80 in Iowa Coun
    0.9877  [not hate] CLINTON, Iowa ( AP ) Police say the boyfriend of a Clinton woman killed in her 
    0.9872  [not hate] Thor Soderberg , 43 , was shot and killed at 61st and Racine .


## 7. Validation — Train-time Self-exclusion

For 3 randomly sampled training tweets, verify that:
1. The tweet's own text does **not** appear in the retrieved neighbors when `chunk_id` is passed.
2. It **does** appear (as the top result) when `chunk_id=None` is passed — confirming it was in the index.

In [22]:
df = pd.read_csv("chunks/chunks_RoBERTa_tokenizer_55_tokens_limit.csv")

def strip_prefix(text):
    return text.replace("[hate] ", "", 1).replace("[not hate] ", "", 1)

sample = df.sample(3, random_state=42)

for _, row in sample.iterrows():
    raw = strip_prefix(row.text)
    cid = int(row.chunk_id)

    without_exclusion = retrieve_top_k(raw, chunk_id=None,  k=K + 1)
    with_exclusion    = retrieve_top_k(raw, chunk_id=cid,   k=K)

    self_in_without = any(text == row.text for text, _ in without_exclusion)
    self_in_with    = any(text == row.text for text, _ in with_exclusion)

    print(f"\n{'='*70}")
    print(f"chunk_id={cid}  |  {raw[:80]}")
    print(f"  Self present WITHOUT exclusion : {self_in_without}  (expected True)")
    print(f"  Self present WITH    exclusion : {self_in_with}   (expected False)")
    assert self_in_without, f"chunk_id={cid} not found in index at all — check index"
    assert not self_in_with, f"Self-exclusion failed for chunk_id={cid}"

print("\nAll self-exclusion checks passed.")

Encoding: 100%|██████████| 1/1 [00:00<00:00, 24.52it/s]


chunk_id=52049  |  "the british airways-american people should use those houses , not immigrants ."
  Self present WITHOUT exclusion : False  (expected True)
  Self present WITH    exclusion : False   (expected False)


AssertionError: chunk_id=52049 not found in index at all — check index